# Download the ERA5 inputs

This notebook downloads the ERA5 hourly single-level files used by notebooks 02 and 05. Run it from the repository root.

Before running it, create a Climate Data Store account, accept the dataset terms and configure `cdsapi` using the [official API instructions](https://cds.climate.copernicus.eu/how-to-api). The API credentials remain in the user's local configuration and are not stored in this notebook.

Existing files are skipped. A first download can take several hours because the requests cover three cities, four variables and thirteen years.

In [ ]:
from pathlib import Path

import cdsapi

DATASET = "reanalysis-era5-single-levels"
OUTDIR = Path("data/raw/era5")
OUTDIR.mkdir(parents=True, exist_ok=True)
client = cdsapi.Client()

CITY_BOXES = {
    "paris": [48.99, 2.21, 48.74, 2.51],
    "lyon": [45.88, 4.72, 45.62, 5.02],
    "marseille": [43.55, 5.20, 43.20, 5.55],
}
VARIABLES = {
    "wind_u": "10m_u_component_of_wind",
    "wind_v": "10m_v_component_of_wind",
    "blh": "boundary_layer_height",
    "t2m": "2m_temperature",
}
MONTHS = [f"{month:02d}" for month in range(1, 13)]
DAYS = [f"{day:02d}" for day in range(1, 32)]
HOURS = [f"{hour:02d}:00" for hour in range(24)]


In [ ]:
def retrieve(filename, request):
    """Download one file unless it is already present."""
    destination = OUTDIR / filename
    if destination.exists():
        print(f"{filename} already present, skipping")
        return
    print(f"Requesting {filename}")
    client.retrieve(DATASET, request, str(destination))
    print(f"{filename} done")


# Notebook 02 uses both wind components in one Paris file for 2013-2024.
for year in range(2013, 2025):
    retrieve(
        f"era5_paris_wind_{year}.nc",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
            ],
            "year": str(year),
            "month": MONTHS,
            "day": DAYS,
            "time": HOURS,
            "area": [49.0, 2.0, 48.7, 2.6],
            "data_format": "netcdf",
        },
    )

# Notebook 05 uses a land-sea mask and four separate variables.
for city, box in CITY_BOXES.items():
    retrieve(
        f"era5_{city}_lsm.nc",
        {
            "product_type": "reanalysis",
            "variable": "land_sea_mask",
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": "00:00",
            "area": box,
            "data_format": "netcdf",
        },
    )

for city, box in CITY_BOXES.items():
    for year in range(2013, 2026):
        for short_name, variable in VARIABLES.items():
            retrieve(
                f"era5_{city}_{short_name}_{year}.nc",
                {
                    "product_type": "reanalysis",
                    "variable": variable,
                    "year": str(year),
                    "month": MONTHS,
                    "day": DAYS,
                    "time": HOURS,
                    "area": box,
                    "data_format": "netcdf",
                },
            )

print("All ERA5 files are present.")
